# Imports and initialization

In [2]:
import re

import logging
import os
from pathlib import Path
import pandas as pd

import spacy
from spacy import displacy
from spacy.tokens.doc import Doc

# defining paths for notebook
current_dir = Path(globals()["_dh"][0])

csv_folder = os.path.join(current_dir, r"csv")
train_csv = "train.csv"

# logging configuration
logging.basicConfig(level=logging.DEBUG)

## Reading csv

In [3]:
# We are not using title, but we'd like to, add it here
used_columns = ["id", "text", "rating"]

df = pd.read_csv(os.path.join(csv_folder, train_csv), usecols=used_columns)

text_test = df["text"][10]

df.head()

,id,text,rating
0,17578,"Por incrível que pareça, para uma bebida desti...",5
1,18658,"O readset pode até ser bom, mais tem outros fo...",1
2,28477,"Foi difícil terminar esse livro , demorou mese...",2
3,43638,"A bola é boa divertida, mas não é nem um pouco...",2
4,26130,Comprei errado! Não tenho leitor de e-books. Q...,1


# Cleaning text

In [4]:
def clean_text(text):
    text = (
        text.lower()
    )  # NOTE: talvez seja interessante deixar alguns caracteres em maiúsculo

    # remove URLs
    text = re.sub(r"http\\S+|www\\S+", " ", text)

    # keeps letters and spaces
    text = re.sub(
        r"[^a-zà-úâêîôûãõç\\s]", " ", text
    )  # NOTE: está tirando os números também

    # removes extra spaces
    text = re.sub(r"\\s+", " ", text).strip()

    return text

In [5]:
print(clean_text(text_test))

ótimo custo benefício  não tampa tão bem o sol  mas pelo valor é ok  enteguega o que propõe


# Features

In [6]:
# 19-23
#
# 24-31
#
# tabela 4 ignora
#
# 24
#
# 46
#
# tabela 6 twitter

## Part-Of-Speech (POS) features

Don't forget to `python -m spacy download pt_core_news_sm`
See https://universaldependencies.org/u/pos/ for information on Tags

In [7]:
nlp = spacy.load("pt_core_news_sm")

In [8]:
# default text
doc_test = nlp(text_test)

In [19]:
# Feature 1
def number_of_adj(doc: Doc):
    logging.debug("[number_of_adj]")

    adj_num = 0
    for token in doc:
        if token.pos_ == "ADJ":
            adj_num += 1

            logging.debug(token.text + "\t|" + token.pos_)
    return adj_num


test = nlp("Que homem bonito")
print(test, "\nNumber of adj:", number_of_adj(test))

DEBUG:root:[number_of_adj]
DEBUG:root:bonito	|ADJ


Que homem bonito 
Number of adj: 1


In [22]:
# Feature 3
def number_of_adv(doc: Doc):
    adv_num = 0

    for token in doc:
        if token.pos_ == "ADV":
            adv_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return adv_num


test = nlp("Ele é muito bonito")
print(test, "\nNumber of adv:", number_of_adv(test))

DEBUG:root:muito	|ADV


Ele é muito bonito 
Number of adv: 1


In [23]:
# Feature 6
# NOTE: eu acho que essa feature é meio nada a ver -vini
def number_of_det(doc: Doc):
    det_num = 0

    for token in doc:
        if token.pos_ == "DET":
            det_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return det_num


print(doc_test, "\nNumber of det:", number_of_det(doc_test))

DEBUG:root:o	|DET


Ótimo custo benefício, não tampa tão bem o sol, mas pelo valor é ok. Enteguega o que propõe. 
Number of det: 1


In [24]:
# Feature 7
def number_of_intj(doc: Doc):
    intj_num = 0

    for token in doc:
        if token.pos_ == "INTJ":
            intj_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return intj_num


test = nlp("Ah, que droga!")
print(test, "\nNumber of intj:", number_of_intj(test))

DEBUG:root:Ah	|INTJ


Ah, que droga! 
Number of intj: 1


In [25]:
# Feature 9
def number_of_num(doc: Doc):
    num_num = 0

    for token in doc:
        if token.pos_ == "NUM":
            num_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return num_num


test = nlp("Dois pesos, 2 medidas")
print(test, "\nNumber of num:", number_of_num(test))

DEBUG:root:Dois	|NUM
DEBUG:root:2	|NUM


Dois pesos, 2 medidas 
Number of num: 2


In [26]:
# Feature 12
def number_of_punct(doc: Doc):
    punct_num = 0

    for token in doc:
        if token.pos_ == "PUNCT":
            punct_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return punct_num


test = nlp("Ah, que droga!")
print(test, "\nNumber of punct:", number_of_punct(test))

DEBUG:root:,	|PUNCT
DEBUG:root:!	|PUNCT


Ah, que droga! 
Number of punct: 2


In [27]:
# Feature 14
def number_of_sym(doc: Doc):
    sym_num = 0

    for token in doc:
        if token.pos_ == "SYM":
            sym_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return sym_num


test = nlp("R$50,00")
print(test, "\nNumber of sym:", number_of_sym(test))

DEBUG:root:R$	|SYM


R$50,00 
Number of sym: 1


In [42]:
# Feature 16 & 18
comp_sup_list = ["melhor", "pior", "maior", "menor"]


def number_of_comp_sup(text: str):
    """Comparative or superlative words"""
    text = text.lower()
    comp_sub_count = 0

    for word in comp_sup_list:
        comp_sub_count += text.count(word)

    return comp_sub_count


test = "A maior e melhor torta é a de limão, e a pior? Não sei"
print(number_of_comp_sup(test))

3


In [35]:
# Feature 18
# NOTE: This feature doesn't seem that meaningful -vini
# NOTE: Couldn't test it
def number_of_x(doc: Doc):
    """X means other"""
    x_num = 0

    for token in doc:
        if token.pos_ == "X":
            x_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return x_num


test = nlp("E aí ele xfgh pdl jklw sombrero fastfood meaningful")
print(test, "\nNumber of x:", number_of_x(test))

E aí ele xfgh pdl jklw sombrero fastfood meaningful 
Number of x: 0


### spaCy testing

In [13]:
doc = nlp(text_test)
for token in doc:
    print(token.text, "\t\t|", token.pos_, token.dep_)

Ótimo 		| PROPN nsubj
custo 		| VERB nsubj
benefício 		| ADJ obj
, 		| PUNCT punct
não 		| ADV advmod
tampa 		| VERB parataxis
tão 		| ADV advmod
bem 		| ADV advmod
o 		| DET det
sol 		| NOUN nsubj
, 		| PUNCT punct
mas 		| CCONJ cc
pelo 		| ADP case
valor 		| NOUN conj
é 		| AUX cop
ok 		| NOUN ROOT
. 		| PUNCT punct
Enteguega 		| VERB ROOT
o 		| PRON obj
que 		| PRON nsubj
propõe 		| VERB acl:relcl
. 		| PUNCT punct


In [14]:
displacy.render(doc, style="dep", options={"compact": "True"})